In [3]:
# Ante Budimir – minimal pipeline (FBref + Sorare)
import pandas as pd
import numpy as np

from fbref import fbref_module as fbref

In [8]:
# Scrape match logs
fbref_url = "https://fbref.com/en/players/8f3565b3/matchlogs/2024-2025/{}/Ante-Budimir-Match-Logs"
stat_url_list = ['summary', 'passing', 'passing_types', 'gca', 'defense', 'possession', 'misc']

fbref_df = fbref.format_column_names(fbref.scrape(fbref_url.format(stat_url_list[0]), 'matchlogs_all'))
for stat_url in stat_url_list[1:]:
    fbref_df_part = fbref.format_column_names(fbref.scrape(fbref_url.format(stat_url), 'matchlogs_all'))
    fbref_df = pd.merge(fbref_df, fbref_df_part,
                        on="Date", suffixes=['', '_remove'])
    fbref_df = fbref_df.drop(columns=[col for col in fbref_df if '_remove' in col])

In [33]:
# Clean
if 'Match Report' in fbref_df.columns:
    fbref_df = fbref_df.drop(columns='Match Report')
fbref_df = fbref_df[(fbref_df.Date != 'Date') & (pd.notna(fbref_df.Date))]

# Szűrés: csak LaLiga meccsek (Competition == 'La Liga')
if "Comp" in fbref_df.columns:
    fbref_df = fbref_df[fbref_df["Comp"] == "La Liga"]

# 0 values
for i in fbref_df.index:
    for col in fbref_df.columns:
        if 'not play' in str(fbref_df.loc[i, col]):
            fbref_df.loc[i, col] = 0

fbref_df.iloc[:, 10:] = fbref_df.iloc[:, 10:].astype(float)

# Dátum normalizálás
fbref_df["Date"] = pd.to_datetime(fbref_df["Date"])
fbref_df["Total_Mis"] = fbref_df["Total_Att"] - fbref_df["Total_Cmp"]

print(fbref_df.head())

        Date  Day     Comp        Round Venue Result    Squad        Opponent  \
0 2024-08-17  Sat  La Liga  Matchweek 1  Home  D 1–1  Osasuna         Leganés   
1 2024-08-24  Sat  La Liga  Matchweek 2  Home  W 1–0  Osasuna        Mallorca   
2 2024-08-29  Thu  La Liga  Matchweek 3  Away  L 0–4  Osasuna          Girona   
3 2024-09-01  Sun  La Liga  Matchweek 4  Home  W 3–2  Osasuna      Celta Vigo   
6 2024-09-16  Mon  La Liga  Matchweek 5  Away  L 1–3  Osasuna  Rayo Vallecano   

  Start Pos  ... Aerial Duels_Lost Aerial Duels_Won% Sorare_Score  \
0     Y  FW  ...               2.0              80.0         46.0   
1     Y  FW  ...               7.0              22.2         35.0   
2     Y  FW  ...               1.0              75.0         35.5   
3     Y  FW  ...               1.0              80.0         70.0   
6     N  FW  ...               2.0               0.0         25.0   

  General_Score Defending_Score Possession_Score Passing_Score  \
0           0.0             0.0 

In [36]:
def calculate_sorare_points(starting_player, position, decisive_actions, negative_actions, all_around_stats):
    weights = pd.read_excel('sorare_weights.xlsx')

    ds = 35 if starting_player else 25
    ds_levels = [25, 35, 60, 70, 80, 90, 100]
    ds_index = ds_levels.index(ds)

    positive_events = decisive_actions.get('goal', 0) + decisive_actions.get('assist', 0) #+ decisive_actions.get('clean_sheet', 0)
    ds_index = min(ds_index + positive_events, len(ds_levels) - 1)

    negative_events = decisive_actions.get('yellow_card', 0) + decisive_actions.get('red_card', 0) + negative_actions.get('own_goal', 0) + negative_actions.get('penalty_miss', 0)
    ds_index = max(ds_index - negative_events, 0)
    
    ds = ds_levels[int(ds_index)] 

    pos = position.upper()
    category_scores = {'General':0, 'Defending':0, 'Possession':0, 'Passing':0, 'Attacking':0, 'Goalkeeping':0}

    for _, row_weights in weights.iterrows():
        cat, stat, w_gk, w_def, w_mid, w_for = row_weights['CATEGORY'], row_weights['STAT'], row_weights['GK'], row_weights['DEF'], row_weights['MID'], row_weights['FOR']
        weight = (w_gk if pos=='GK' else w_def if pos=='DEF' else w_mid if pos=='MID' else w_for)
        value = all_around_stats.get(stat, 0)
        product = value * weight
        if product != 0:
            print(f'{cat}, {stat}, {weight:.2f}*{value:.2f}={product:.2f}')
        category_scores[cat] += product
    
    aa = max(0, sum(category_scores.values()))


    return ds + aa, category_scores

# --- Példa ---
starting_player = True
position = 'FOR'
decisive_actions = {'goal': 0, 'assist': 0, 'clean_sheet': 0, 'yellow_card': 0, 'red_card': 0}
negative_actions = {'own_goal': 0, 'penalty_miss': 0}
all_around_stats = {
    'shot_on_target': 1, 'won_contest': 1, 'penalty_area_entry': 1,
    'fouls': 1, 'was_fouled': 0, 'errors_leading_to_shot': 0,
    'possession_lost': 8, 'duel_lost': 4, 'duel_won': 1, 'interception': 1,
    'accurate_pass': 12, 'successful_final_third_passes': 9
}

points = calculate_sorare_points(starting_player, position, decisive_actions, negative_actions, all_around_stats)
print(f"\nVégső Sorare pontszám: {points}")


General, fouls, -0.50*1.00=-0.50
Possession, possession_lost, -0.10*8.00=-0.80
Possession, duel_lost, -1.00*4.00=-4.00
Possession, duel_won, 1.00*1.00=1.00
Possession, interception, 3.00*1.00=3.00
Passing, accurate_pass, 0.10*12.00=1.20
Passing, successful_final_third_passes, 0.10*9.00=0.90
Attacking, shot_on_target, 3.00*1.00=3.00
Attacking, won_contest, 0.50*1.00=0.50
Attacking, penalty_area_entry, 0.50*1.00=0.50

Végső Sorare pontszám: (39.8, {'General': -0.5, 'Defending': 0.0, 'Possession': -0.7999999999999998, 'Passing': 2.1, 'Attacking': 4.0, 'Goalkeeping': 0.0})


In [42]:
# Hozzon létre üres listákat a pontszámok tárolására
sorare_scores = []
category_scores_list = []

# Iteráljon végig az fbref_df DataFrame sorain
for index, row in fbref_df.iterrows():
    print('')
    print(row["Opponent"])
    position = row.get('Pos', 'FOR') 
    starting_player = row.get('Start', '') == 'Y'

    all_around_stats = {
        'yellow_card': row.get('Performance_CrdY', 0),
        'fouls': row.get('Performance_Fls', 0),
        'was_fouled': row.get('Performance_Fld', 0),
        'error_lead_to_shot': row.get('Err', 0), 
        'effective_clearance': row.get('Clr', 0),
        'won_tackle': row.get('Tackles_TklW', 0),
        'blocked_cross': row.get('Blocks_Blocks', 0),
        'outfielder_block': row.get('Blocks_Sh', 0),
        'possession_lost': row.get('Total_Mis', 0) + row.get('Carries_Mis', 0),
        'duel_lost': row.get('Aerial Duels_Lost', 0), 
        'duel_won': row.get('Aerial Duels_Won', 0), 
        'interception': row.get('Performance_Int', 0),
        'big_chance_created': row.get('SCA_GCA', 0),
        'big_chance_missed': row.get('Standard_PKatt', 0) - row.get('Standard_PK', 0),
        'adjusted_total_att_assist': row.get('Expected_xAG', 0),
        'accurate_pass': row.get('Passes_Cmp', 0),
        'successful_final_third_passes': row.get('Passes_PrgP', 0),
        'accurate_long_balls': row.get('Long_Cmp', 0),
        'missed_pass': row.get('Passes_Att', 0) - row.get('Passes_Cmp', 0),
        'shot_on_target': row.get('Performance_SoT', 0),
        'penalty_area_entry': row.get('Carries_1/3', 0),
    }

    decisive_actions = {
        'goal': row.get('Performance_Gls', 0),
        'assist': row.get('Performance_Ast', 0),
        'clean_sheet': 1 if row.get('Min', 0) >= 60 and row.get('Performance_Gls', 0) == 0 else 0,
        'yellow_card': 1 if row.get('Performance_CrdY', 0) > 0 else 0,
        'red_card': 1 if row.get('Performance_CrdR', 0) > 0 else 0
    }
    
    negative_actions = {
        'own_goal': row.get('Performance_OG', 0),
        'penalty_miss': 1 if row.get('Performance_PKatt', 0) > 0 and row.get('Performance_PK', 0) == 0 else 0
    }

    # Sorare pontszám és kategória pontszámok kiszámítása a függvény segítségével
    points, scores = calculate_sorare_points(
        starting_player, 
        position, 
        decisive_actions, 
        negative_actions, 
        all_around_stats
    )
    
    # Hozzáadása a listákhoz
    sorare_scores.append(points)
    category_scores_list.append(scores)

# Hozzon létre egy új oszlopot a DataFrame-ben a teljes pontszámokkal
fbref_df['Sorare_Score'] = sorare_scores

# Hozzon létre új oszlopokat a DataFrame-ben a kategória pontszámokkal
for category in category_scores_list[0].keys():
    fbref_df[f'{category}_Score'] = [scores.get(category, 0) for scores in category_scores_list]

print(fbref_df[['Date', 'Opponent', 'Sorare_Score', 'General_Score', 'Defending_Score', 'Possession_Score', 'Passing_Score', 'Attacking_Score', 'Goalkeeping_Score']].head())
print(f'\nMean: {np.array(sorare_scores[-15:]).mean():.2f}\nStd: {np.array(sorare_scores[-15:]).std():.2f}')


Leganés
General, fouls, -0.50*2.00=-1.00
General, was_fouled, 1.00*1.00=1.00
Possession, possession_lost, -0.10*15.00=-1.50
Possession, duel_lost, -1.00*2.00=-2.00
Possession, duel_won, 1.00*8.00=8.00
Passing, accurate_pass, 0.10*15.00=1.50
Passing, successful_final_third_passes, 0.10*1.00=0.10
Attacking, shot_on_target, 3.00*1.00=3.00
Attacking, penalty_area_entry, 0.50*2.00=1.00

Mallorca
Possession, possession_lost, -0.10*8.00=-0.80
Possession, duel_lost, -1.00*7.00=-7.00
Possession, duel_won, 1.00*2.00=2.00
Passing, accurate_pass, 0.10*4.00=0.40
Attacking, shot_on_target, 3.00*1.00=3.00

Girona
General, fouls, -0.50*3.00=-1.50
Possession, possession_lost, -0.10*4.00=-0.40
Possession, duel_lost, -1.00*1.00=-1.00
Possession, duel_won, 1.00*3.00=3.00
Passing, accurate_pass, 0.10*1.00=0.10

Celta Vigo
General, fouls, -0.50*1.00=-0.50
Possession, possession_lost, -0.10*5.00=-0.50
Possession, duel_lost, -1.00*1.00=-1.00
Possession, duel_won, 1.00*4.00=4.00
Possession, interception, 3.00